## Machine Learning Enigneer Workflows:

![mle workflow diagram](images/mle_workflow.png)

**Workflow Steps**
> *Depending on the dataset, the question we’re trying to answer and the tech stack we’re working with, one or more of these steps can be omitted or combined with another.*

1. ETL (Extract, Transform, load) data
2. Data cleaning
3. Train-test-validation Split
4. EDA (Exploratory Data Analysis)
5. Feature Engineering
    - (Normalization, removing autocorrelations, discretization)
6. Model Selection and implementation
7. Model Evaluation
8. Hyperparameter Tuning
9. Model Validation
10. Build ML pipeline

---

**Extract, Transform, Load Data (ETL):**

- Depending on the volume of data, an engineer would use a tool like PySpark to extract this data, transform it and load it into a local database.


**Data Cleaning and Aggregation:**

- Tasks include: dealing with null or missing entries, conforming timestamps to a standard, carrying out aggregations like grouping events based on timestamps by the hour or day, grouping IP’s by location, etc.


**Train-Test-Validation Split:**

```python
from sklearn.model_selection import train_test_split

# For feature matrix X and target variable y
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)
``` 
- `random_state` : ensures reproducibility, by reusing **random_state**, certain dataset would be split in the exact manner.
- The training data is used to learn a model’s parameters and the test data is used to test its performance.
- models in production, a third portion of the dataset is set aside; known as a **holdout or validation dataset** used to tune hyperparameters and/or to perform model validation later on.

![train-test split](images/train-test_split_diagram.png)

> *The reason we don’t want to perform data manipulations before splitting the dataset into training, test and validation datasets is that we don’t want data points from one of these to influence the other. Suppose we needed a machine learning model to predict housing prices and wanted to standardize a feature like the size of an apartment, the average of this value would look different between the entire dataset and each of the individual datasets. Scaling the entire column to the average value misses the unique information contained within the subpopulations and will make the model evaluation and validation process less objective.*


**Exploratory Data Analysis (EDA):**

- the step of inspecting, analyzing and altering your data to get it ready for machine learning modeling.
- the step where decisions on how to deal with outliers, transform are made.
    - preprocessing
    - imputation 
    - feature selection
    - dimensionality reduction.

**Feature Engineering**

- Feature engineer refers to an umbrella of methods to prep, select and reduce features in a machine learning problem. 
> *Feature engineering can also involve using machine learning algorithms like PCA to reduce dimensionality or methods that are implemented during the model fitting step like regularization.*


**Model Selection and Implementation**

- The choice of the model depends on the attributes of the data., as well as the type of question to be answered.


**Model Evaluation**

-  Whatever model is built, it must be evaluated on the test data.
> *For `classification problems`, metrics like **accuracy, precision, recall, F1 score and AUROC scores** indicate how performant the model is and for `regression problems`, scores like **RMSE and R-squared** are some commonly used metrics.*

> **Machine learning engineers iterate over different types of models to figure out the most optimal model for the problem at hand.**


**Hyperperameter Tuning**

- Once a model has been decided upon, it can be tuned for better performance.
- Hyperparameter tuning is essential in making sure that the model does not overfit or underfit the data.

> *This is key to how well the model is fitting known data and how well it’s able to generalize to new data as well. Hence hyperparameter tuning might be done on the validation or holdout dataset.*


**Model Validation**
- Model validation is the process of making sure that the model is still performant on data that it hasn’t seen at all.
    - (neither in the training phase nor in the test phase.)

> *This can be done either during the hyperparameter tuning step or after. Typically the same metrics used during the model evaluation phase needs to be used here as well so as to make a reasonable comparison with the former.*


**Build ML Pipeline**
- A ML pipeline is a modular sequence of objects that codifies and automates a ML workflow to make it efficient, reproducible and generalizable.

---


## Pipeline

`scikit-learn Pipeline` : chain together the different steps that go into the ML workflow.

- pipelines provide consistency (same steps, same order, same conditions)

1. To define a pipeline, pass a list of tuples of the form `(name, transform/estimator)` into a pipeline object. 

```python
# pipeline(task 1-> task 2-> task 3)
from sklearn.pipeline import Pipeline
pipeline = Pipeline([('importer', SimpleImporter()), ('scale', StandardScaler())])
```

2. Once a pipeline object has been instantiated, the methods `.fit` and `.transform` can be called like we would with any data transformation. 

```python
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(X,y, random_state=0, test_size=0.25)
pipeline.fit(x_train)
pipeline.transform(x_test)
```

> *If the pipeline includes a machine learning model as well,`.predict` can also be called down the line.*

> *Each step in the pipeline will be fit in the order provided. Further parameters can be passed to each step as well.*
    ```python
    # to pass the parameter with_mean=False to the StandardScaler:
    Pipeline([("imputer",SimpleImputer()), ("scale",StandardScaler(with_mean=False))])
    ```
---

### Pipeline : Data Cleaning (Numeric)

```python
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

## Loading the dataset
columns = ["sex","length","diam","height","whole","shucked","viscera","shell","age"]
df = pd.read_csv("http://archive.ics.uci.edu/ml/machine-learning-databases/abalone/abalone.data",names=columns)
## Defining target and predictor variables
y = df.age
X = df.drop(columns=['age'])

## Numerical columns:
num_cols = X.select_dtypes(include=np.number).columns
## Categorical columns
cat_cols = X.select_dtypes(include=['object']).columns

## Create some missing values
for i in range(1000):
    X.loc[np.random.choice(X.index),np.random.choice(X.columns)] = np.nan

## Perform train-test split
x_train, x_test, y_train, y_test = train_test_split(X,y, random_state=0, test_size=0.25)

#####-------Imputation and Scaling: Code base to transform -----------------#####
## Numerical training data
x_train_num = x_train[num_cols]
# Filling in missing values with mean on numeric features only
x_train_fill_missing = x_train_num.fillna(x_train_num.mean())
## Fitting standard scaler on x_train_fill_missing
scale = StandardScaler().fit(x_train_fill_missing)
## Scaling data after filling in missing values
x_train_fill_missing_scale = scale.transform(x_train_fill_missing)
## Same steps as above, but on the test set:
x_test_fill_missing = x_test[num_cols].fillna(x_train_num.mean())
x_test_fill_missing_scale = scale.transform(x_test_fill_missing)
#####-------Imputation and Scaling: Code base to transform -----------------#####

#1. Rewrite using Pipelines!
pipeline = Pipeline([('imputer', SimpleImputer()), ('scaler', StandardScaler(with_mean=True))])

#2. Fit pipeline on the test and compare results
pipeline.fit(x_train[num_cols])
x_transform = pipeline.transform(x_test[num_cols])

#3.  Verify pipeline transform test set is the same by using np.array_equal()
array_diff = np.array_equal(x_transform, x_test_fill_missing_scale)
print(array_diff)

#4. Change imputer strategy to median 
pipeline_median = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])

pipeline_median.fit(x_train[num_cols])
x_transform_median = pipeline_median.transform(x_test[num_cols])

# 5 Compare results between the two pipelines
new_array_diff = np.abs(x_transform - x_transform_median).sum().round(4)
print(new_array_diff)
```
---

### Pipeline : Data Cleaning (Categorical)

```python
import numpy as np
import pandas as pd
from sklearn import datasets
from sklearn.model_selection import KFold, train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline


columns = ["sex","length","diam","height","whole","shucked","viscera","shell","age"]
df = pd.read_csv("http://archive.ics.uci.edu/ml/machine-learning-databases/abalone/abalone.data",names=columns)

y = df.age
X=df.drop(columns=['age'])
num_cols = X.select_dtypes(include=np.number).columns
cat_cols = X.select_dtypes(include=['object']).columns
#create some missing values
for i in range(1000):
    X.loc[np.random.choice(X.index),np.random.choice(X.columns)] = np.nan

x_train, x_test, y_train, y_test = train_test_split(X,y, random_state=0, test_size=0.25)
x_train_cat = x_train[cat_cols]
#fill missing values with mode on categorical features only
x_train_fill_missing = x_train_cat.fillna(x_train_cat.mode().values[0][0])
#apply one hot encoding on x_train_fill_missing
ohe = OneHotEncoder(sparse=False, drop='first').fit(x_train_fill_missing)
#transform data after filling in missing values
x_train_fill_missing_ohe = ohe.transform(x_train_fill_missing)

#Now want to do the same thing on the test set! 
x_test_fill_missing = x_test[cat_cols].fillna(x_train_cat.mode().values[0][0])
x_test_fill_missing_ohe = ohe.transform(x_test_fill_missing)

#1. Rewrite using Pipelines!
pipeline = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(drop='first', sparse=False))])


#2. Fit the pipeline and transform the test data (categorical columns only!)
pipeline.fit(x_train[cat_cols], y_train)
x_transform = pipeline.transform(x_test[cat_cols])

#3. Check if the two arrays are the same using np.array_equal()

check_arrays = np.array_equal(x_transform, x_test_fill_missing_ohe)

print('Are the arrays equal?')
print(check_arrays)
```

---

### Pipeline: Column Transformer

`ColumnTransformer()` : to simply apply every function to all columns. Combines the pipeline processes together for certain columns of different data types.

- Takes in a list of tuples (name, pipeline, columns)
- Can be anythin with a `.fit` and `.transform` method, i.e (SimpleImputer, StandardScaler)
- Can also be a pipeline itself.

```python
import numpy as np
import pandas as pd
from sklearn import datasets
from sklearn.model_selection import KFold, train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

columns = ["sex","length","diam","height","whole","shucked","viscera","shell","age"]
df = pd.read_csv("http://archive.ics.uci.edu/ml/machine-learning-databases/abalone/abalone.data",names=columns)

y = df.age
X=df.drop(columns=['age'])
num_cols = X.select_dtypes(include=np.number).columns
cat_cols = X.select_dtypes(include=['object']).columns
#create some missing values
for i in range(1000):
    X.loc[np.random.choice(X.index),np.random.choice(X.columns)] = np.nan

x_train, x_test, y_train, y_test = train_test_split(X,y, random_state=0, test_size=0.25)

#1. Create numerical and categorical pipelines called `num_vals` and `cat_vals`
num_vals = Pipeline([('imputer', SimpleImputer(strategy='mean')), ('scale', StandardScaler())])

cat_vals = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(drop='first', sparse=False))])

#2. Create the column transformer with the categorical and numerical processes
preprocess = ColumnTransformer(
  transformers = [
    ('num_preprocess', num_vals, num_cols),
    ('cat_preprocess', cat_vals, cat_cols)
    ])

#3. Fit the preprocess transformer to training data
preprocess.fit(x_train, y_train)
x_transform = preprocess.transform(x_test)
```
---



### Pipeline: Adding a Model

- The last step is the only step in the pipeline that can be non-transformer. 


```python
import numpy as np
import pandas as pd

from sklearn import svm, datasets
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score

columns = ["sex","length","diam","height","whole","shucked","viscera","shell","age"]
df = pd.read_csv("http://archive.ics.uci.edu/ml/machine-learning-databases/abalone/abalone.data",names=columns)

y = df.age
X=df.drop(columns=['age'])
num_cols = X.select_dtypes(include=np.number).columns
cat_cols = X.select_dtypes(include=['object']).columns
#create some missing values
for i in range(1000):
    X.loc[np.random.choice(X.index),np.random.choice(X.columns)] = np.nan

x_train, x_test, y_train, y_test = train_test_split(X,y, random_state=0, test_size=0.25)

cat_vals = Pipeline([("imputer",SimpleImputer(strategy='most_frequent')), ("ohe",OneHotEncoder(sparse=False, drop='first'))])

num_vals = Pipeline([("imputer",SimpleImputer(strategy='mean')), ("scale",StandardScaler())])

preprocess = ColumnTransformer(
    transformers=[
        ("cat_process", cat_vals, cat_cols),
        ("num_process", num_vals, num_cols)
    ]
)
#1. Create a pipeline with `preprocess` and a linear regression model, `regr`
pipeline = Pipeline([('preprocess', preprocess), ('regr', LinearRegression())])

#2. Fit the pipeline on the training data and predict on the test data
pipeline.fit(x_train, y_train)
y_pred = pipeline.predict(x_test)


#3. Calculate pipeline score and compare to estimator score
#Pipeline score
pipeline_score = pipeline.score(x_test, y_test)
print(pipeline_score)

#r-squared score
r2_score = r2_score(y_test, y_pred)
print(r2_score)
```

---

### Pipeline: Hyperperameter tuning

- tune some of the parameters of the model by applying a grid search over a range of hyperparameter values.

- pipeline is an estimator and can call the `.fit()` and `.predict()` methods on it. 

- the whole pipeline can be passed as an estimator for GridSearchCV. This will then refit the pipeline for each combination of parameter values in the grid and each fold in the cross-validation split.

> *any hyperparameter can be called using `pipeline_step_name + '__' + hyperparameter`. For example, `regr__fit_intercept` corresponds to a pipeline step named “regr” and the hyperparameter “fit_intercept”.*

```python
import numpy as np
import pandas as pd

from sklearn import svm, datasets
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, GridSearchCV

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn import metrics

columns = ["sex","length","diam","height","whole","shucked","viscera","shell","age"]
df = pd.read_csv("http://archive.ics.uci.edu/ml/machine-learning-databases/abalone/abalone.data",names=columns)

y = df.age
X=df.drop(columns=['age'])
num_cols = X.select_dtypes(include=np.number).columns
cat_cols = X.select_dtypes(include=['object']).columns
#create some missing values
for i in range(1000):
    X.loc[np.random.choice(X.index),np.random.choice(X.columns)] = np.nan

x_train, x_test, y_train, y_test = train_test_split(X,y, random_state=0, test_size=0.25)

cat_vals = Pipeline([("imputer",SimpleImputer(strategy='most_frequent')), ("ohe",OneHotEncoder(sparse=False, drop='first'))])
num_vals = Pipeline([("imputer",SimpleImputer(strategy='mean')), ("scale",StandardScaler())])

preprocess = ColumnTransformer(
    transformers=[
        ("cat_process", cat_vals, cat_cols),
        ("num_process", num_vals, num_cols)
    ]
)

#Create a pipeline with pregrocess and a linear regression model
pipeline = Pipeline([("preprocess",preprocess), 
                     ("regr",LinearRegression())])

#Very simple parameter grid, with and without the intercept
param_grid = {
    "regr__fit_intercept": [True,False]
}

#------------------------------------------------
#1. Grid search using previous pipeline
gs = GridSearchCV(estimator=pipeline, param_grid=param_grid, scoring='neg_mean_squared_error', cv=5)

#2. fit grid using training data and print best score
gs.fit(x_train, y_train)
best_score = gs.best_score_
best_params = gs.best_params_
print('best score: ', best_score)
print('best params: ', best_params)
```
---

### Pipeline: Final Pipeline

```python
import numpy as np
import pandas as pd

from sklearn import svm, datasets
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.model_selection import train_test_split, GridSearchCV

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn import metrics

columns = ["sex","length","diam","height","whole","shucked","viscera","shell","age"]
df = pd.read_csv("http://archive.ics.uci.edu/ml/machine-learning-databases/abalone/abalone.data",names=columns)

y = df.age
X=df.drop(columns=['age'])
num_cols = X.select_dtypes(include=np.number).columns
cat_cols = X.select_dtypes(include=['object']).columns
#create some missing values
for i in range(1000):
    X.loc[np.random.choice(X.index),np.random.choice(X.columns)] = np.nan

x_train, x_test, y_train, y_test = train_test_split(X,y, random_state=0, test_size=0.25)

cat_vals = Pipeline([("imputer",SimpleImputer(strategy='most_frequent')), ("ohe",OneHotEncoder(sparse=False, drop='first'))])
num_vals = Pipeline([("imputer",SimpleImputer(strategy='mean')), ("scale",StandardScaler())])

preprocess = ColumnTransformer(
    transformers=[
        ("cat_preprocess", cat_vals, cat_cols),
        ("num_preprocess", num_vals, num_cols)
    ]
)
#Create a pipeline with preprocess and a linear regression model
pipeline = Pipeline([("preprocess",preprocess), 
                     ("regr",LinearRegression())])

#--------------------------------------------------------------
# 1. Update the `search_space` array from the narrative to add a Lasso Regression model as the third dictionary.
search_space = [
  {'regr':[LinearRegression()], 'regr__fit_intercept': [True,False]}, 
  {'regr':[Ridge()], 'regr__alpha':[0.01,0.1,1,10,100]},
  {'regr': [Lasso()], 'regr__alpha':[0.01,0.1,1,10,100]}]


# 2.  Initialize a grid search on `search_space`
gs = GridSearchCV(pipeline, search_space, scoring='neg_mean_squared_error', cv=5)

#3. Find the best pipeline, regression model and its hyperparameters

## i. Fit to training data
gs.fit(x_train, y_train)

## ii. Find the best pipeline
best_pipeline = gs.best_estimator_

## iii. Find the best regression model
best_regression_model = best_pipeline.named_steps['regr']
print('The best regression model is:')
print(best_regression_model)

## iv. Find the hyperparameters of the best regression model
best_model_hyperparameters= best_regression_model.get_params()

print('The hyperparameters of the regression model are:')
print(best_model_hyperparameters)

#4. Access the hyperparameters of the categorical preprocessing step
cat_preprocess_hyperparameters=best_pipeline.named_steps['preprocess'].named_transformers_['cat_preprocess'].named_steps['imputer'].get_params()

print('The hyperparameters of the imputer are:')
print(cat_preprocess_hyperparameters)
```

---


### Takeaways

- Pipelines help make concise, reproducible, code by combining steps of transformers and/or a final estimator.

- Intermediate steps of a pipeline must have both the .fit() and .transform() methods. This includes preprocessing, imputation, feature selection, dimension reduction.

- The final step of a pipeline must have the .fit() method – this can include a transformer or an estimator/model.

- If the pipeline is meant to only transform your data by combining preprocessing and data cleaning steps, then each step in the pipeline will be a transformer. If your pipeline will also include a model (a final estimation or prediction step), then the last step must be an estimator.

- Once the steps of a pipeline are defined, it can be used like an other transformer/estimator by calling fit, transform, and/or predict methods. Similarly, it can be used in place of an estimator in a hyperparameter grid search.

### Pipeline: Custom Classes

```python
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import KFold, train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin

columns = ["sex","length","diam","height","whole","shucked","viscera","shell","age"]
df = pd.read_csv("http://archive.ics.uci.edu/ml/machine-learning-databases/abalone/abalone.data",names=columns)
y = df.age
X=df.drop(columns=['age'])
num_cols = X.select_dtypes(include=np.number).columns
cat_cols = X.select_dtypes(include=['object']).columns

for i in range(1000):
    X.loc[np.random.choice(X.index),np.random.choice(X.columns)] = np.nan
x_train, x_test, y_train, y_test = train_test_split(X,y, random_state=0, test_size=0.25)
x_train_num = x_train[num_cols]
#fill missing values with mean on numeric features only
x_train_fill_missing = x_train_num.fillna(x_train_num.mean())
#fit standard scaler on x_train_fill_missing
scale = StandardScaler().fit(x_train_fill_missing)
#scale data after filling in missing values
x_train_fill_missing_scale = scale.transform(x_train_fill_missing)
x_test_fill_missing = x_test[num_cols].fillna(x_train_num.mean())
x_test_fill_missing_scale = scale.transform(x_test_fill_missing)

class MyImputer(BaseEstimator, TransformerMixin): 
    def __init__(self):
        return None
    
    def fit(self, X, y = None):
        self.means = np.mean(X, axis=0)    # calculate the mean of each column
        return self
    
    def transform(self, X, y = None):
        #transform method fills in missing values with means using pandas
        return X.fillna(self.means)

#1. Create new pipeline using the custom class MyImputer as the first step and standard scaler on the second
new_pipeline = Pipeline([('imputer', MyImputer()), ('scaler', StandardScaler())])

#2. Fit new pipeline on the training data with num_cols only and verify that the results of the transform are the same on test set
new_pipeline.fit(x_train[num_cols], y_train)
x_transform = new_pipeline.transform(x_test[num_cols])

check_arrays=np.array_equal(x_transform, x_test_fill_missing_scale)

print(check_arrays)
```
